# DartSpot: YOLO26 mit dem Original-Datensatz

Die Läufe v5 und v6 haben das laufende Modell (v4, YOLOv8n) nicht verbessert, weil v4 auf den ~16 000 Bildern des **DeepDarts-Datensatzes** (McNally et al. 2021) aufbaut und unsere ~1100 Bilder allein dagegen nichts Neues beibringen. Dieses Notebook probiert deshalb das, was bisher fehlt: **ein neueres Modell von Grund auf trainieren** (YOLO26, vortrainiert auf COCO) – und zwar auf **DeepDarts (3500 zufällige Bilder daraus) plus unseren Bildern** – und prüft, ob es die schwachen Fälle (Double-Ring, Rand) besser trifft. Zwei Größen laufen nacheinander: **yolo26n** (so klein wie v4, aber neuer) und **yolo26s** (3-4x so rechenintensiv).

Vorher in Colab: **Laufzeit → Laufzeittyp ändern → GPU (T4)**, danach **Alle ausführen** und die Zip \`training-set-rectified-v7.zip\` hochladen. Ein Durchlauf dauert etwa 30-60 Minuten.

Was am Ende zählt, wie bisher: Abstand zur echten Spitze in Millimetern, getrennt nach Double, Treble/Rand, links unten. Neu: **Rechenzeit auf einem CPU-Kern** (die App rechnet im Browser auf dem Handy) und die **Form der ONNX-Ausgabe** (die App versteht das klassische Format \`[1, 9, N]\` und das neue NMS-freie \`[1, K, 6]\`).

In [ ]:
!pip -q install ultralytics
from google.colab import drive, files
import os, shutil, zipfile

drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/DeepDarts-Dataset/"
# Neuestes Modell zuerst: v4 ist das, was gerade in der App läuft; tip ist der Stand davor.
# Der Lauf, der das heutige Modell (v4) erzeugt hat, hat es unter dem alten Namen dartspot_tip_best.pt gesichert - die Datei
# heisst also "tip", enthaelt aber v4 (das, was in der App laeuft). v5 war ein Lauf ohne Verbesserung, wird NICHT genommen.
# Neuere Laeufe sichern unter dartspot_v6_best.pt usw. und werden nur dann hier eingetragen, wenn sie in die App gehen.
CANDIDATES = [DRIVE + "dartspot_tip_best.pt", DRIVE + "dartspot_v4_best.pt"]
ORIGINAL_PT = DRIVE + "training_runs/dart_yolov8n/weights/best.pt"
START_PT = next((p for p in CANDIDATES if os.path.exists(p)), None)
if START_PT:
    shutil.copy(START_PT, "/content/best.pt")  # Drive-Datei bleibt unberührt
    print("Start vom aktuellen Modell aus Drive:", os.path.basename(START_PT))
else:
    print("Kein gesichertes Modell in Drive gefunden. Bitte die best.pt des aktuell laufenden Modells hochladen (Downloads, die .pt, nicht die .onnx):")
    up = files.upload()
    pts = [n for n in up if n.lower().endswith(".pt")]
    if pts:
        shutil.copy(pts[0], "/content/best.pt")
        print("Start von der hochgeladenen best.pt.")
    else:
        assert os.path.exists(ORIGINAL_PT), "Weder dartspot_tip_best.pt noch eine hochgeladene .pt noch das Original gefunden"
        shutil.copy(ORIGINAL_PT, "/content/best.pt")
        print("ACHTUNG: Start vom ORIGINAL-Modell (nicht vom aktuellen Spitzen-Modell).")

DRIVE_ZIP = DRIVE + "training-set-rectified-v7.zip"
if os.path.exists(DRIVE_ZIP):
    zip_names = [DRIVE_ZIP]   # Kopie liegt schon in Drive: kein Upload noetig
    print("Datenset aus Drive:", DRIVE_ZIP)
else:
    print("Lade training-set-rectified-v7.zip hoch (Tipp: einmal in den Drive-Ordner DeepDarts-Dataset legen, dann entfaellt der Upload):")
    uploaded = files.upload()
    zip_names = [n for n in uploaded if n.lower().endswith(".zip")]
assert zip_names, "Keine .zip hochgeladen"
shutil.rmtree("/content/ds", ignore_errors=True)  # frisch entpacken, auch bei erneutem Ausführen
zipfile.ZipFile(zip_names[0]).extractall("/content/ds")
# Zips made with Windows PowerShell store paths with backslashes; move such files into real folders.
for name in os.listdir("/content/ds"):
    if "\\" in name:
        target = os.path.join("/content/ds", name.replace("\\", "/"))
        os.makedirs(os.path.dirname(target), exist_ok=True)
        shutil.move(os.path.join("/content/ds", name), target)
DS = "/content/ds" if os.path.exists("/content/ds/data.yaml") else "/content/ds/training-set-rectified"
with open(f"{DS}/data.yaml", "w") as f:
    f.write("\n".join([f"path: {DS}", "train: images/train", "val: images/val", "names:",
                       "  0: dart", "  1: cal_1", "  2: cal_2", "  3: cal_3", "  4: cal_4", ""]))
print("Datensatz:", DS, os.listdir(DS))

## Messung: Abstand zur echten Dartspitze
Das Board hat im entzerrten Bild einen Radius von 0,377 der Bildbreite = 170 mm. Daraus rechnen wir Pixelabstände in Millimeter um. Verglichen wird die Mitte der gefundenen Box mit dem Spitzen-Label (jeder echte Dart wird dem nächstgelegenen Fund zugeordnet).

In [ ]:
import glob, math
from ultralytics import YOLO

MM_PER_UNIT = 170 / 0.377   # Millimeter je Bildbreite im entzerrten Bild

def read_darts(label_file):
    out = []
    for ln in open(label_file).read().splitlines():
        p = ln.split()
        if p and p[0] == "0":
            out.append((float(p[1]), float(p[2])))
    return out

RADIUS = 0.377  # Boardradius im entzerrten Bild (Anteil der Bildbreite)

def hard_kind(g):
    """Schwierige Darts, gemessen an Martins Runden: Double/Rand, Treble, links unten."""
    dx, dy = g[0] - 0.5, g[1] - 0.5
    r = math.hypot(dx, dy) / RADIUS
    if r > 0.93:
        return "double"
    if (0.57 < r < 0.66) or r > 0.88:
        return "treble/rand"
    if dx < 0 and dy > 0 and r > 0.6:
        return "links unten"
    return None

def tip_metrics(model, split="val", conf=0.12, dense_only=False, kind=None):
    # conf=0.12 wie in der App: dort zaehlt auch ein schwacher Fund, wenn er auf veraenderten Pixeln liegt
    errors, n_pred, n_gt = [], 0, 0
    for img in sorted(glob.glob(f"{DS}/images/{split}/*.jpg")):
        gt = read_darts(img.replace("/images/", "/labels/").rsplit(".", 1)[0] + ".txt")
        if kind:
            gt = [g for g in gt if hard_kind(g) == kind]
            if not gt:
                continue
        if dense_only:  # only darts that have another dart within 25 mm next to them (tight groups)
            gt = [g for g in gt if any(o is not g and math.hypot(o[0] - g[0], o[1] - g[1]) * MM_PER_UNIT < 25 for o in gt)]
            if not gt:
                continue
        res = model.predict(img, imgsz=800, conf=conf, verbose=False)[0]
        preds = [(float(b.xywhn[0][0]), float(b.xywhn[0][1])) for b in res.boxes if int(b.cls) == 0]
        n_pred += len(preds); n_gt += len(gt)
        free = preds[:]
        for g in gt:
            if not free:
                errors.append(float("inf")); continue
            j = min(range(len(free)), key=lambda k: math.hypot(free[k][0] - g[0], free[k][1] - g[1]))
            errors.append(math.hypot(free[j][0] - g[0], free[j][1] - g[1]) * MM_PER_UNIT)
            free.pop(j)
    finite = sorted(e for e in errors if e != float("inf"))
    within = lambda mm: sum(1 for e in errors if e <= mm) / max(1, len(errors))
    return {
        "darts": n_gt, "found": len(finite), "extra_detections": max(0, n_pred - len(finite)),
        "median_mm": finite[len(finite) // 2] if finite else None,
        "r8": within(8.5), "r17": within(17), "r34": within(34),
    }

def ap50_by_class(metrics):
    return {int(c): float(a) for c, a in zip(metrics.box.ap_class_index, metrics.box.ap50)}

def show(name, m):
    med = "-" if m["median_mm"] is None else f"{m['median_mm']:.1f} mm"
    print(f"{name:14s} gefunden {m['found']}/{m['darts']}  Median {med:>9s}  "
          f"<=8.5mm {m['r8']*100:4.0f}%  <=17mm {m['r17']*100:4.0f}%  <=34mm {m['r34']*100:4.0f}%  zusaetzliche Funde {m['extra_detections']}")

## Vorher-Messung: das laufende Modell (ALT) – Genauigkeit und Rechenzeit

In [ ]:
import time, numpy as np, onnxruntime as ort, os

def time_onnx(path, runs=7):
    """Median-Rechenzeit einer Auswertung auf EINEM CPU-Kern (grobe Vergleichszahl, kein Handy-Wert)."""
    so = ort.SessionOptions(); so.intra_op_num_threads = 1
    sess = ort.InferenceSession(path, so, providers=["CPUExecutionProvider"])
    name = sess.get_inputs()[0].name
    x = np.random.rand(1, 3, 800, 800).astype(np.float32)
    sess.run(None, {name: x})  # warm-up
    times = []
    for _ in range(runs):
        t0 = time.perf_counter(); sess.run(None, {name: x}); times.append((time.perf_counter() - t0) * 1000)
    out_shape = [o.shape for o in sess.get_outputs()]
    return float(np.median(times)), out_shape, os.path.getsize(path) / 1e6

old = YOLO("/content/best.pt")
KINDS = ["double", "treble/rand", "links unten"]
old_tip = tip_metrics(old)
old_dense = tip_metrics(old, dense_only=True)
old_hard = {k: tip_metrics(old, kind=k) for k in KINDS}
old_ap = ap50_by_class(old.val(data=f"{DS}/data.yaml", imgsz=800, plots=False))
old_onnx = old.export(format="onnx", imgsz=800, opset=12, simplify=True, dynamic=False)
old_ms, old_shape, old_mb = time_onnx(old_onnx)
show("ALT", old_tip); show("ALT dicht", old_dense)
for k in KINDS: show("ALT " + k, old_hard[k])
print(f"ALT Rechenzeit {old_ms:.0f} ms (1 CPU-Kern), {old_mb:.1f} MB, Ausgabe {old_shape}")

## Original-Datensatz einmischen (DeepDarts)
Die Bilder liegen in deinem Drive-Ordner (cropped_images.zip + labels_pkl.zip, ~16 000 Bilder, 800 px). Das Notebook zieht 3500 zufällige Bilder heraus und schreibt die Labels im selben Format wie unsere (Klasse 0 = Dart, 1-4 = die vier Ecken). **Sicherheitsnetz:** welcher der vier Eck-Punkte im Original zu welcher Klasse gehört, prüft das Notebook selbst, indem es die Zuordnung wählt, bei der das laufende Modell (v4) auf den Bildern am besten übereinstimmt. Bewertet wird weiterhin NUR auf unseren 72 echten Bewertungsbildern.

In [ ]:
import itertools, random, pandas as pd

DD_SAMPLE = 3500
BOX_DART, BOX_CORNER = 0.04, 0.05
dd_zips = sorted(glob.glob(DRIVE + "cropped_images*.zip"))
dd_pkls = sorted(glob.glob(DRIVE + "labels_pkl*.zip")) + sorted(glob.glob(DRIVE + "labels.pkl"))
MIX = DS
if not dd_zips or not dd_pkls:
    print("ACHTUNG: DeepDarts-Dateien (cropped_images.zip / labels_pkl.zip) nicht in", DRIVE, "- es wird nur mit unseren Bildern trainiert.")
else:
    t0 = time.time()
    pkl_path = dd_pkls[0]
    if pkl_path.endswith(".zip"):
        zipfile.ZipFile(pkl_path).extractall("/content/ddlabels"); pkl_path = "/content/ddlabels/labels.pkl"
    df = pd.read_pickle(pkl_path)
    print("Spalten:", list(df.columns), "| Zeilen:", len(df))
    shutil.copy(dd_zips[0], "/content/cropped_images.zip")   # schneller Zugriff als direkt vom Drive
    zf = zipfile.ZipFile("/content/cropped_images.zip")
    members = {n.split("/", 2)[-1] if n.startswith("cropped_images/800/") else n: n for n in zf.namelist() if n.lower().endswith(".jpg")}
    def find_member(folder, name):
        for key in (f"{folder}/{name}", f"{folder}/{name}".replace("\\", "/")):
            if key in members: return members[key]
        return None
    rows = df.sample(n=min(DD_SAMPLE, len(df)), random_state=0)
    items = []
    for _, row in rows.iterrows():
        member = find_member(str(row["img_folder"]), str(row["img_name"]))
        try:
            arr = np.array([np.asarray(p, dtype=float).ravel()[:2] for p in row["xy"]])
        except Exception:
            continue
        pts = arr if arr.ndim == 2 and arr.shape[1] == 2 else np.zeros((0, 2))
        if member is None or len(pts) < 4 or not np.isfinite(pts[:4]).all():
            continue
        items.append((member, pts))
    print("Bilder mit Labels gefunden:", len(items), "von", len(rows))

    # 1. Zuordnung der vier Eck-Punkte zu den Klassen cal_1..cal_4 vom laufenden Modell lernen lassen
    probe = items[:40]
    for member, _ in probe:
        zf.extract(member, "/content/ddprobe")
    cost = np.zeros((4, 4))   # cost[i, j]: Abstand des Original-Punkts i zur Vorhersage der Klasse j+1
    for member, pts in probe:
        res = old.predict(f"/content/ddprobe/{member}", imgsz=800, conf=0.3, verbose=False)[0]
        best = {}
        for b in res.boxes:
            c = int(b.cls)
            if 1 <= c <= 4 and (c not in best or float(b.conf) > best[c][0]):
                best[c] = (float(b.conf), float(b.xywhn[0][0]), float(b.xywhn[0][1]))
        for i in range(4):
            for j in range(4):
                if (j + 1) in best:
                    cost[i, j] += math.hypot(pts[i][0] - best[j + 1][1], pts[i][1] - best[j + 1][2])
                else:
                    cost[i, j] += 0.5
    perm = min(itertools.permutations(range(4)), key=lambda p: sum(cost[i, p[i]] for i in range(4)))
    mean_err = sum(cost[i, perm[i]] for i in range(4)) / (4 * len(probe))
    print("Zuordnung Original-Punkt -> Klasse:", {i: perm[i] + 1 for i in range(4)}, f"| mittlere Abweichung {mean_err:.3f} (Bildbreite)")
    assert mean_err < 0.08, f"Eck-Zuordnung passt nicht zum laufenden Modell (Abweichung {mean_err:.3f}); Beispielzeile: {df.iloc[0].to_dict()} - bitte an Claude schicken"

    # 2. Bilder und Labels schreiben
    os.makedirs("/content/dd/images/train", exist_ok=True); os.makedirs("/content/dd/labels/train", exist_ok=True)
    written = 0
    for k, (member, pts) in enumerate(items):
        zf.extract(member, "/content/ddraw")
        name = f"dd{k:05d}"
        shutil.move(f"/content/ddraw/{member}", f"/content/dd/images/train/{name}.jpg")
        lines = [f"{perm[i] + 1} {pts[i][0]:.6f} {pts[i][1]:.6f} {BOX_CORNER} {BOX_CORNER}" for i in range(4)]
        for x, y in pts[4:]:
            if 0 < x < 1 and 0 < y < 1:
                lines.append(f"0 {x:.6f} {y:.6f} {BOX_DART} {BOX_DART}")
        open(f"/content/dd/labels/train/{name}.txt", "w").write("
".join(lines) + "
")
        written += 1
    shutil.rmtree("/content/ddraw", ignore_errors=True); shutil.rmtree("/content/ddprobe", ignore_errors=True)
    os.makedirs("/content/mix", exist_ok=True)
    MIX = "/content/mix"
    with open(f"{MIX}/data.yaml", "w") as f:
        f.write("
".join([f"path: /content", f"train: [{DS}/images/train, /content/dd/images/train]", f"val: {DS}/images/val", "names:",
                           "  0: dart", "  1: cal_1", "  2: cal_2", "  3: cal_3", "  4: cal_4", ""]))
    print(f"DeepDarts-Bilder eingemischt: {written} | Trainingsbilder insgesamt: {len(glob.glob(DS + '/images/train/*.jpg')) + written} | {time.time() - t0:.0f} s")

## Training: YOLO26 n und s
Kein Spiegeln/Drehen (die 20 ist im entzerrten Bild immer oben, die Ecken haben feste Plätze), kein Mosaic (es würde das Board zerstückeln und die Ecken-Klassen zerstören). Geringe Farb- und Verschiebungs-Augmentierung wie bisher. Frühes Stoppen nach 15 Epochen ohne Besserung. Mit dem Original-Datensatz dauert eine Epoche etwa eine Minute (n) bzw. zwei (s): rechne mit 1-2 Stunden insgesamt.

In [ ]:
VARIANTS = {
    "yolo26n": dict(weights="yolo26n.pt", epochs=45, batch=16),
    "yolo26s": dict(weights="yolo26s.pt", epochs=35, batch=16),
}
results = {}
for name, cfg in VARIANTS.items():
    m = YOLO(cfg["weights"])
    m.train(
        data=f"{MIX}/data.yaml", imgsz=800, epochs=cfg["epochs"], batch=cfg["batch"], patience=15, optimizer="AdamW", lr0=0.001, lrf=0.05,
        fliplr=0.0, flipud=0.0, degrees=0.0, translate=0.05, scale=0.15, shear=0.0, perspective=0.0,
        mosaic=0.0, mixup=0.0, hsv_h=0.01, hsv_s=0.5, hsv_v=0.35,
        project="/content/runs", name=name, exist_ok=True, plots=False,
    )
    best_path = str(m.trainer.best)
    trained = YOLO(best_path)
    r = {"path": best_path, "tip": tip_metrics(trained), "dense": tip_metrics(trained, dense_only=True),
         "hard": {k: tip_metrics(trained, kind=k) for k in KINDS},
         "ap": ap50_by_class(trained.val(data=f"{DS}/data.yaml", imgsz=800, plots=False))}
    # export the way the app needs it; the classic raw head ([1, 9, N]) first, the NMS-free head as a fallback
    onnx_path, mode = None, None
    for kwargs, label in ((dict(end2end=False), "klassisch [1,9,N]"), (dict(), "NMS-frei (Standard)")):
        try:
            onnx_path = trained.export(format="onnx", imgsz=800, opset=12, simplify=True, dynamic=False, **kwargs); mode = label; break
        except Exception as err:
            print(name, "Export", label, "fehlgeschlagen:", str(err)[:160])
    if onnx_path:
        r["ms"], r["shape"], r["mb"] = time_onnx(onnx_path)
        r["onnx"], r["mode"] = onnx_path, mode
    results[name] = r
    show(name, r["tip"]); show(name + " dicht", r["dense"])
    for k in KINDS: show(name + " " + k, r["hard"][k])
    print(f"{name}: Rechenzeit {r.get('ms', float('nan')):.0f} ms (1 CPU-Kern, ALT {old_ms:.0f} ms), {r.get('mb', 0):.1f} MB, Ausgabe {r.get('shape')} ({r.get('mode')})")

## Ergebnis und Entscheidung
Ein neues Modell geht nur in die App, wenn es **spürbar besser** ist (mindestens +5 Prozentpunkte innerhalb 17 mm insgesamt **oder** +15 bei den Doubles), die vier Ecken nicht schlechter sind und es **nicht mehr als 2,5x so langsam** ist wie das laufende.

In [ ]:
show("ALT", old_tip)
for k in KINDS: show("ALT " + k, old_hard[k])
verdicts = {}
for name, r in results.items():
    print()
    show(name, r["tip"])
    for k in KINDS: show(name + " " + k, r["hard"][k])
    print(f"   AP50 je Klasse: { {c: round(a, 3) for c, a in r['ap'].items()} }  |  {r.get('ms', float('nan')):.0f} ms vs ALT {old_ms:.0f} ms")
    corners_ok = all(r["ap"].get(c, 0) >= old_ap.get(c, 0) - 0.05 for c in (1, 2, 3, 4))
    better = (r["tip"]["r17"] >= old_tip["r17"] + 0.05) or (r["hard"]["double"]["r17"] >= old_hard["double"]["r17"] + 0.15)
    fast_enough = r.get("ms", 1e9) <= 2.5 * old_ms
    verdicts[name] = better and corners_ok and fast_enough
    print(f"   -> besser: {better} | Ecken ok: {corners_ok} | schnell genug: {fast_enough}  =>  {'JA' if verdicts[name] else 'NEIN'}")

winners = [n for n, ok in verdicts.items() if ok]
print()
if winners:
    WINNER = max(winners, key=lambda n: results[n]["tip"]["r17"])
    print("Gewinner:", WINNER, "- JA, in die App")
else:
    WINNER = max(results, key=lambda n: results[n]["tip"]["r17"])
    print("Kein Modell erfüllt alle Bedingungen - NEIN, altes Modell behalten (bestes Experiment:", WINNER + ")")

## Export
Die ONNX-Datei des besten Experiments zum Herunterladen (auch bei NEIN, damit ich sie offline gegen das laufende Modell messen kann). Das Original in Drive bleibt unberührt.

In [ ]:
from google.colab import files
win = results[WINNER]
shutil.copy(win["path"], "/content/drive/MyDrive/DeepDarts-Dataset/dartspot_yolo26_best.pt")
print("ONNX:", win.get("onnx"), "| Modus:", win.get("mode"), "| Ausgabeform:", win.get("shape"))
if win.get("onnx"):
    files.download(win["onnx"])
files.download(win["path"])